# OutlineCompilerFunctionsWithExistingGlobalSymbols 和 MarkCompilerFunctionsAsExtern 外部代码生成

用于外部代码生成的两个重要 pass 的测试：
1. `OutlineCompilerFunctionsWithExistingGlobalSymbols`：将带有特定编译器标记的函数提取到全局命名空间
2. `MarkCompilerFunctionsAsExtern`：将特定编译器的函数标记为外部函数

In [1]:
import set_env

In [5]:
import tvm
import tvm.testing
import numpy as np

def make_const(dtype, shape):
    """创建指定数据类型和形状的 TVM Relay 常量"""
    return tvm.relay.const(np.random.rand(*shape).astype(dtype))

def make_consts(dtype, shapes):
    """创建多个指定数据类型和形状的 TVM Relay 常量"""
    return [make_const(dtype, shape) for shape in shapes]

# 常量表，存储测试中使用的常量
metatable = {
    "relay.Constant": make_consts(
        "float16",
        [
            (2304, 768),  # 0: 矩阵乘法的权重
            (2304,),      # 1: 偏置向量
            (600, 32, 64),  # 2: 批处理矩阵乘法的输入
        ],
    )
}

In [6]:
def original_mod():
    """生成原始的 Relay 模块，包含带有 Cutlass 和 cuBLAS 编译器标记的函数"""
    return tvm.relay.parse(
        """
        #[version = "0.0.5"]
        def @main(%x0 : Tensor[(1600, 768), float16], %x3 : Tensor[(600, 32, 64), float16]) -> (Tensor[(1600, 2304), float16], Tensor[(600, 32, 32), float16]) {
          %0 = fn(%y_0_i0: Tensor[(1600, 768), float16], %y_0_i1: Tensor[(2304, 768), float16], %y_0_i2: Tensor[(2304), float16],
                  Inline=1, Compiler="cutlass", global_symbol="tvmgen_default_cutlass_main_0", Primitive=1) -> Tensor[(1600, 2304), float16] {
            %4 = fn (%FunctionVar_0_0: Tensor[(1600, 768), float16], %FunctionVar_0_1: Tensor[(2304, 768), float16], %FunctionVar_0_2: Tensor[(2304), float16],
                     PartitionedFromPattern="nn.dense_add_", Composite="cutlass.dense_bias") -> Tensor[(1600, 2304), float16] {
              %5 = nn.dense(%FunctionVar_0_0, %FunctionVar_0_1, units=2304);
              add(%5, %FunctionVar_0_2)
            };
            %4(%y_0_i0, %y_0_i1, %y_0_i2)
          };
          %1 = %0(%x0, meta[relay.Constant][0], meta[relay.Constant][1]);
          %2 = fn(%y_3_i0: Tensor[(600, 32, 64), float16], %y_3_i1: Tensor[(600, 32, 64), float16],
                  Inline=1, Compiler="cublas", global_symbol="tvmgen_default_cublas_main_3", Primitive=1) -> Tensor[(600, 32, 32), float16] {
            %6 = fn (%FunctionVar_0_01: Tensor[(600, 32, 64), float16], %FunctionVar_0_11: Tensor[(600, 32, 64), float16],
                     PartitionedFromPattern="nn.batch_matmul_", Composite="cublas.batch_matmul") -> Tensor[(600, 32, 32), float16] {
              nn.batch_matmul(%FunctionVar_0_01, %FunctionVar_0_11, out_dtype="float16", transpose_b=True)
            };
            %6(%y_3_i0, %y_3_i1)
          };
          %3 = %2(%x3, meta[relay.Constant][2]);
          (%1, %3)
        }
        """,
        "from_string",
        None,
        metatable,
    )


In [8]:
ori_mod = original_mod()
ori_mod.show()

正确地从主函数中提取 Cutlass 编译器标记的函数到全局命名空间：

In [9]:
actual_outlined_mod = tvm.relay.transform.OutlineCompilerFunctionsWithExistingGlobalSymbols(
    "cutlass"
)(original_mod())

In [10]:
actual_outlined_mod.show()

验证 let 绑定的 Cutlass 编译器标记函数是否正确地从主函数中提取到全局命名空间：

In [11]:
def original_mod_let_bound():
    """生成 let-bound 形式的原始 Relay 模块，用于测试 let 绑定函数"""
    return tvm.relay.parse(
        """
        #[version = "0.0.5"]
        def @main(%x0 : Tensor[(1600, 768), float16], %x3 : Tensor[(600, 32, 64), float16]) -> (Tensor[(1600, 2304), float16], Tensor[(600, 32, 32), float16]) {
          let %f = fn(%y_0_i0: Tensor[(1600, 768), float16], %y_0_i1: Tensor[(2304, 768), float16], %y_0_i2: Tensor[(2304), float16],
                      Inline=1, Compiler="cutlass", global_symbol="tvmgen_default_cutlass_main_0", Primitive=1) -> Tensor[(1600, 2304), float16] {
            %4 = fn (%FunctionVar_0_0: Tensor[(1600, 768), float16], %FunctionVar_0_1: Tensor[(2304, 768), float16], %FunctionVar_0_2: Tensor[(2304), float16],
                     PartitionedFromPattern="nn.dense_add_", Composite="cutlass.dense_bias") -> Tensor[(1600, 2304), float16] {
              %5 = nn.dense(%FunctionVar_0_0, %FunctionVar_0_1, units=2304);
              add(%5, %FunctionVar_0_2)
            };
            %4(%y_0_i0, %y_0_i1, %y_0_i2)
          };
          %1 = %f(%x0, meta[relay.Constant][0], meta[relay.Constant][1]);
          %2 = fn(%y_3_i0: Tensor[(600, 32, 64), float16], %y_3_i1: Tensor[(600, 32, 64), float16],
                  Inline=1, Compiler="cublas", global_symbol="tvmgen_default_cublas_main_3", Primitive=1) -> Tensor[(600, 32, 32), float16] {
            %6 = fn (%FunctionVar_0_01: Tensor[(600, 32, 64), float16], %FunctionVar_0_11: Tensor[(600, 32, 64), float16],
                     PartitionedFromPattern="nn.batch_matmul_", Composite="cublas.batch_matmul") -> Tensor[(600, 32, 32), float16] {
              nn.batch_matmul(%FunctionVar_0_01, %FunctionVar_0_11, out_dtype="float16", transpose_b=True)
            };
            %6(%y_3_i0, %y_3_i1)
          };
          %3 = %2(%x3, meta[relay.Constant][2]);
          (%1, %3)
        }
        """,
        "from_string",
        None,
        metatable,
    )


actual_outlined_mod = tvm.relay.transform.OutlineCompilerFunctionsWithExistingGlobalSymbols(
    "cutlass"
)(original_mod_let_bound())

In [12]:
actual_outlined_mod.show()

验证外部的 Cutlass 编译器标记函数是否正确地标记为外部函数（Extern=1）：

In [16]:
def expected_extern_mod():
    """将带有 Cutlass 编译器标记的全局函数标记为外部函数（Extern=1）"""
    return tvm.relay.parse(
        """
        #[version = "0.0.5"]
        def @main(%x0 : Tensor[(1600, 768), float16], %x3 : Tensor[(600, 32, 64), float16]) -> (Tensor[(1600, 2304), float16], Tensor[(600, 32, 32), float16]) {
          %1 = @tvmgen_default_cutlass_main_0(%x0, meta[relay.Constant][0], meta[relay.Constant][1]);
          %2 = fn(%y_3_i0: Tensor[(600, 32, 64), float16], %y_3_i1: Tensor[(600, 32, 64), float16],
                  Inline=1, Compiler="cublas", global_symbol="tvmgen_default_cublas_main_3", Primitive=1) -> Tensor[(600, 32, 32), float16] {
            %6 = fn (%FunctionVar_0_01: Tensor[(600, 32, 64), float16], %FunctionVar_0_11: Tensor[(600, 32, 64), float16],
                     PartitionedFromPattern="nn.batch_matmul_", Composite="cublas.batch_matmul") -> Tensor[(600, 32, 32), float16] {
              nn.batch_matmul(%FunctionVar_0_01, %FunctionVar_0_11, out_dtype="float16", transpose_b=True)
            };
            %6(%y_3_i0, %y_3_i1)
          };
          %3 = %2(%x3, meta[relay.Constant][2]);
          (%1, %3)
        }

        def @tvmgen_default_cutlass_main_0(%y_0_i0: Tensor[(1600, 768), float16], %y_0_i1: Tensor[(2304, 768), float16], %y_0_i2: Tensor[(2304), float16],
                  Extern=1) -> Tensor[(1600, 2304), float16] {
          %4 = fn (%FunctionVar_0_0: Tensor[(1600, 768), float16], %FunctionVar_0_1: Tensor[(2304, 768), float16], %FunctionVar_0_2: Tensor[(2304), float16],
                   PartitionedFromPattern="nn.dense_add_", Composite="cutlass.dense_bias") -> Tensor[(1600, 2304), float16] {
            %5 = nn.dense(%FunctionVar_0_0, %FunctionVar_0_1, units=2304);
            add(%5, %FunctionVar_0_2)
          };
          %4(%y_0_i0, %y_0_i1, %y_0_i2)
        }
        """,
        "from_string",
        None,
        metatable,
    )

actual_extern_mod = tvm.relay.transform.MarkCompilerFunctionsAsExtern("cutlass")(
    expected_extern_mod()
)

In [17]:
actual_extern_mod.show()

验证指定的全局函数是否正确地内联回主函数体中：

In [19]:
def expected_outlined_mod():
    """将带有 Cutlass 编译器标记的函数从主函数体中提取到全局命名空间"""
    return tvm.relay.parse(
        """
        #[version = "0.0.5"]
        def @main(%x0 : Tensor[(1600, 768), float16], %x3 : Tensor[(600, 32, 64), float16]) -> (Tensor[(1600, 2304), float16], Tensor[(600, 32, 32), float16]) {
          %1 = @tvmgen_default_cutlass_main_0(%x0, meta[relay.Constant][0], meta[relay.Constant][1]);
          %2 = fn(%y_3_i0: Tensor[(600, 32, 64), float16], %y_3_i1: Tensor[(600, 32, 64), float16],
                  Inline=1, Compiler="cublas", global_symbol="tvmgen_default_cublas_main_3", Primitive=1) -> Tensor[(600, 32, 32), float16] {
            %6 = fn (%FunctionVar_0_01: Tensor[(600, 32, 64), float16], %FunctionVar_0_11: Tensor[(600, 32, 64), float16],
                     PartitionedFromPattern="nn.batch_matmul_", Composite="cublas.batch_matmul") -> Tensor[(600, 32, 32), float16] {
              nn.batch_matmul(%FunctionVar_0_01, %FunctionVar_0_11, out_dtype="float16", transpose_b=True)
            };
            %6(%y_3_i0, %y_3_i1)
          };
          %3 = %2(%x3, meta[relay.Constant][2]);
          (%1, %3)
        }

        def @tvmgen_default_cutlass_main_0(%y_0_i0: Tensor[(1600, 768), float16], %y_0_i1: Tensor[(2304, 768), float16], %y_0_i2: Tensor[(2304), float16],
                  Inline=1, Compiler="cutlass", global_symbol="tvmgen_default_cutlass_main_0", Primitive=1) -> Tensor[(1600, 2304), float16] {
          %4 = fn (%FunctionVar_0_0: Tensor[(1600, 768), float16], %FunctionVar_0_1: Tensor[(2304, 768), float16], %FunctionVar_0_2: Tensor[(2304), float16],
                   PartitionedFromPattern="nn.dense_add_", Composite="cutlass.dense_bias") -> Tensor[(1600, 2304), float16] {
            %5 = nn.dense(%FunctionVar_0_0, %FunctionVar_0_1, units=2304);
            add(%5, %FunctionVar_0_2)
          };
          %4(%y_0_i0, %y_0_i1, %y_0_i2)
        }
        """,
        "from_string",
        None,
        metatable,
    )
mod = expected_outlined_mod()
gv = mod.get_global_var("tvmgen_default_cutlass_main_0")
actual_inlined_mod = tvm.relay.transform.InlineCompilerFunctionsBoundTo([gv])(mod)

In [20]:
actual_inlined_mod.show()